# Segmentation figures -- SyMTRS

Six models (U-Net, SegFormer b0/b1/b2, DeepLabV3 R50/R101) at four resolutions
(256 / 512 / 1024 / 2048). The figures answer, in order: what does the whole grid look
like (3D), how do the models trade places (slope, bump), does capacity buy accuracy
(parameters), which class pays for low resolution (per-class), and what does that look
like on a tile (qualitative).


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Patch
from matplotlib import colors as mcolors
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)
from PIL import Image

def _figures_dir() -> Path:
    """This folder, whether the code runs as a script or as a notebook.

    A notebook has no `__file__` and starts in whatever directory it was launched from
    (the paper folder, usually), so fall back to looking for this file upwards from there.
    """
    try:
        return Path(__file__).resolve().parent
    except NameError:
        pass
    cwd = Path.cwd().resolve()
    for anchor in (cwd, *cwd.parents):
        for candidate in (anchor / "figures", anchor / "SyMTRS___IJCV_latex" / "figures", anchor):
            if (candidate / "segmentation_figures.py").is_file():
                return candidate
    raise SystemExit(f"cannot find segmentation_figures.py from {cwd}; run from the paper folder")


HERE = _figures_dir()
REPO = HERE.parents[1]
RUNS = REPO / "runs" / "campaign2"
QUAL = Path("/tmp/segqual/ladder")
DATA_ROOT = Path("/mnt/ssd1/SyMTRS")  # read-only; the split the runs were trained on

# ---------------------------------------------------------------------------
# Palette. Validated with the dataviz six-checks against a white surface:
#   categorical (6 slots, adjacent pairs, model series): ALL CHECKS PASS
#   categorical (3 slots, all pairs, model families):    ALL CHECKS PASS
# Three slots sit below 3:1 on white, so every series is directly labelled as well
# (the relief rule) -- no figure here asks the reader to match colour alone.
# ---------------------------------------------------------------------------
SURFACE = "#ffffff"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
HAIR, AXIS = "#e1e0d9", "#c3c2b7"

BLUE_RAMP = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
             "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ = mcolors.LinearSegmentedColormap.from_list("symtrs_blue", BLUE_RAMP)

RESOLUTIONS = [256, 512, 1024, 2048]

PARAMS = {  # million parameters, from the "N params" line each run logs at build
    "unet_small": 7.86, "segformer_b0": 3.72, "segformer_b1": 13.68,
    "segformer_b2": 24.72, "deeplabv3_r50": 42.05, "deeplabv3_r101": 61.10,
}
# Display order = legend order; consecutive slots are the pairs that look most alike.
MODELS = ["unet_small", "segformer_b0", "segformer_b1", "segformer_b2",
          "deeplabv3_r50", "deeplabv3_r101"]
SHORT = {"unet_small": "U-Net", "segformer_b0": "SegFormer b0", "segformer_b1": "SegFormer b1",
         "segformer_b2": "SegFormer b2", "deeplabv3_r50": "DeepLabV3 R50",
         "deeplabv3_r101": "DeepLabV3 R101"}
TINY = {"unet_small": "U-Net", "segformer_b0": "SF-b0", "segformer_b1": "SF-b1",
        "segformer_b2": "SF-b2", "deeplabv3_r50": "DL-R50", "deeplabv3_r101": "DL-R101"}
FAMILY = {"unet_small": "U-Net", "segformer_b0": "SegFormer", "segformer_b1": "SegFormer",
          "segformer_b2": "SegFormer", "deeplabv3_r50": "DeepLabV3", "deeplabv3_r101": "DeepLabV3"}
COLOR = dict(zip(MODELS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
FAM_COLOR = {"U-Net": "#2a78d6", "SegFormer": "#eb6834", "DeepLabV3": "#1baf7a"}

CLASSES = ["building", "vegetation", "road", "car"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": HAIR, "grid.linewidth": 0.7,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 2.0, "lines.markersize": 6.5,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def tidy(ax, grid_axis="y"):
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis=grid_axis)
    ax.grid(axis="x" if grid_axis == "y" else "y", visible=False)


def save(fig, name):
    # pad_inches=0.01: the paper pays for every millimetre of margin, and bbox_inches
    # ="tight" still leaves matplotlib's 0.1 in pad by default.
    fig.savefig(HERE / f"{name}.pdf", bbox_inches="tight", pad_inches=0.01)
    fig.savefig(HERE / f"{name}.png", dpi=200, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)
    print(f"wrote {name}.pdf / .png")
    try:  # show it inline when this runs as a notebook; no-op headless
        from IPython.display import Image as IPImage, display
        display(IPImage(filename=str(HERE / f"{name}.png"), width=760))
    except Exception:
        pass



def spread_labels(ys, min_gap):
    """Push label positions apart, in order, so two near-equal values stay readable."""
    order = np.argsort(ys)
    out = np.array(ys, dtype=float)
    for k in range(1, len(order)):
        prev, cur = order[k - 1], order[k]
        out[cur] = max(out[cur], out[prev] + min_gap)
    return out


In [ ]:
# ---------------------------------------------------------------------------
# Data: the 24 scored runs
# ---------------------------------------------------------------------------
def load_grid():
    out = {}
    for path in sorted(RUNS.glob("*_*/summary.json")):
        m = re.match(r"(.+)_(256|512|1024|2048)$", path.parent.name)
        if not m or m.group(1) not in MODELS:
            continue
        test = json.loads(path.read_text()).get("test") or {}
        if "miou" not in test:
            continue
        out[(m.group(1), int(m.group(2)))] = test
    missing = [(mo, r) for mo in MODELS for r in RESOLUTIONS if (mo, r) not in out]
    assert not missing, f"unscored cells: {missing}"
    return out


DATA = load_grid()
MIOU = np.array([[DATA[(mo, r)]["miou"] for r in RESOLUTIONS] for mo in MODELS])
CAR = np.array([[DATA[(mo, r)]["iou_4"] for r in RESOLUTIONS] for mo in MODELS])

print("test mIoU (rows = models, cols = resolutions)")
for mo, row in zip(MODELS, MIOU):
    print(f"  {mo:<15}", "  ".join(f"{v:.4f}" for v in row))

# The car class itself shrinks in the downsampled labels: at 256 px a car is 2-4 px wide
# and the label resampling fuses neighbours. Part of the low-resolution car IoU drop is
# this, not the model -- the text says so, so the figure prints it.
CAR_FRAC = {}
for r in RESOLUTIONS:
    s = DATA[(MODELS[0], r)]
    tot = sum(s[f"support_{i}"] for i in range(5))
    CAR_FRAC[r] = s["support_4"] / tot
print("\ncar pixels as a fraction of the scored pixels:",
      {r: f"{100 * f:.3f}%" for r, f in CAR_FRAC.items()})


## Figure 1 -- the grid in three dimensions

Two renderings of the same 24 numbers. The ridgeline (1a) reads like a terrain map:
each model is a ridge, height is mIoU, and where the ridges cross is where one model
overtakes another. The surface (1b) is the literal request -- resolution x model x
accuracy -- with the iso-accuracy contours projected on the floor.


In [ ]:
def fig_ridges():
    zmin = 0.78
    fig = plt.figure(figsize=(7.0, 4.5))
    # A 3d axes centres itself in a lot of white space; make it fill the canvas.
    ax = fig.add_axes((0.03, -0.02, 0.92, 0.92), projection="3d")
    ax.set_box_aspect(None, zoom=1.34)
    xs = np.array([0, 1, 2, 3])
    for i, mo in enumerate(MODELS):
        z, c = MIOU[i], COLOR[mo]
        ax.plot([xs[0], xs[0]], [i, i], [zmin, z[0]], color=c, lw=0.8, alpha=0.5)
        ax.plot([xs[-1], xs[-1]], [i, i], [zmin, z[-1]], color=c, lw=0.8, alpha=0.5)
        ax.plot(xs, np.full(4, i), z, color=c, lw=2.4, zorder=10, solid_capstyle="round")
        ax.scatter(xs, np.full(4, i), z, color=c, s=26, depthshade=False, zorder=11,
                   edgecolor=SURFACE, linewidth=0.8)

    ax.set_xticks(xs)
    ax.set_xticklabels(RESOLUTIONS, fontsize=8)
    ax.set_yticks(range(len(MODELS)))
    ax.set_yticklabels([SHORT[m] for m in MODELS], fontsize=8)
    ax.tick_params(axis="y", pad=2)
    ax.set_zlim(zmin, 0.99)
    ax.set_zticks([0.80, 0.85, 0.90, 0.95])
    ax.set_zlabel("test mIoU", labelpad=6)
    ax.set_xlabel("resolution (px)", labelpad=4)
    ax.view_init(elev=22, azim=-64)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 1))
        axis.pane.set_edgecolor(HAIR)
    ax.grid(True)
    save(fig, "seg_fig1a_ridges3d")


def fig_surface():
    fig = plt.figure(figsize=(6.0, 4.6))
    ax = fig.add_subplot(111, projection="3d")
    ax.set_box_aspect(None, zoom=1.26)
    X, Y = np.meshgrid([0, 1, 2, 3], np.arange(len(MODELS)))
    norm = mcolors.Normalize(vmin=0.78, vmax=0.99)
    ax.plot_surface(X, Y, MIOU, facecolors=SEQ(norm(MIOU)), rstride=1, cstride=1,
                    linewidth=0.6, edgecolor=SURFACE, antialiased=True, shade=False)
    ax.contour(X, Y, MIOU, zdir="z", offset=0.78, levels=[0.80, 0.85, 0.90, 0.95],
               cmap=SEQ, linewidths=1.1)
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(RESOLUTIONS, fontsize=8)
    ax.set_yticks(range(len(MODELS)))
    ax.set_yticklabels([TINY[m] for m in MODELS], fontsize=8)
    ax.set_zlim(0.78, 0.985)
    ax.set_zticks([0.80, 0.85, 0.90, 0.95])
    ax.set_zlabel("test mIoU", labelpad=6)
    ax.set_xlabel("resolution (px)", labelpad=4)
    ax.set_ylabel("model", labelpad=6)
    ax.view_init(elev=22, azim=-52)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor(HAIR)
    # No colorbar: it repeated the z axis, and at this size its label printed on top of the
    # z label. The height of the sheet is the scale, and the z ticks read it off.
    save(fig, "seg_fig1b_surface3d")


fig_ridges()
fig_surface()


## Figure 2 -- slope chart

The same matrix as six lines on a log resolution axis. This is the figure that shows
what the table hides: at 256 px the models span 0.118 mIoU, at 2048 px they span 0.012.


In [ ]:
def fig_slope():
    fig, ax = plt.subplots(figsize=(6.4, 3.9))
    xs = np.array(RESOLUTIONS, dtype=float)
    first = MIOU[:, 0]
    label_y = dict(zip(MODELS, spread_labels(list(first), 0.012)))
    for mo in MODELS:
        y = MIOU[MODELS.index(mo)]
        ax.plot(xs, y, color=COLOR[mo], marker="o", markeredgecolor=SURFACE,
                markeredgewidth=1.2, zorder=5)
        ax.annotate(TINY[mo], (xs[0], y[0]), xytext=(xs[0] * 0.90, label_y[mo]),
                    textcoords="data", color=COLOR[mo], fontsize=8.5, va="center", ha="right",
                    fontweight="bold" if mo == "unet_small" else "normal",
                    arrowprops=dict(arrowstyle="-", color=COLOR[mo], lw=0.6,
                                    shrinkA=0, shrinkB=3))
    ax.set_xscale("log", base=2)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{r}\n{'native' if r == 2048 else 'x' + str(2048 // r)}"
                        for r in RESOLUTIONS])
    ax.set_xlim(150, 2300)
    ax.set_ylim(0.775, 0.995)
    ax.set_xlabel("evaluation resolution")
    ax.set_ylabel("test mIoU (4 classes)")
    tidy(ax)
    spread256 = MIOU[:, 0].max() - MIOU[:, 0].min()
    spread2048 = MIOU[:, 3].max() - MIOU[:, 3].min()
    ax.text(0.03, 0.97, f"six models span\n{spread256:.3f} mIoU at 256 px\n"
                        f"{spread2048:.3f} mIoU at 2048 px",
            transform=ax.transAxes, va="top", ha="left", fontsize=8.5, color=INK2,
            linespacing=1.5)
    save(fig, "seg_fig2_slope")


fig_slope()


## Figure 3 -- rank across resolutions (bump chart)

U-Net leads at every level. What moves is the tail: DeepLabV3 is last at 256 px and
fourth at 2048 px, SegFormer b0 starts fourth and ends last.


In [ ]:
def fig_bump():
    ranks = np.array([np.argsort(np.argsort(-MIOU[:, j])) for j in range(4)]).T + 1
    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    xs = np.arange(4)
    for i, mo in enumerate(MODELS):
        y = ranks[i]
        ax.plot(xs, y, color=COLOR[mo], lw=2.4, solid_capstyle="round", zorder=3)
        ax.scatter(xs, y, s=200, color=COLOR[mo], edgecolor=SURFACE, linewidth=1.6, zorder=4)
        for j, r in enumerate(y):
            ax.text(j, r, str(r), color="white" if mo != "segformer_b2" else INK,
                    ha="center", va="center", fontsize=8.5, fontweight="bold", zorder=5)
        ax.annotate(SHORT[mo], (xs[0], y[0]), xytext=(-12, 0), textcoords="offset points",
                    ha="right", va="center", color=COLOR[mo], fontsize=8.5)
    ax.set_xticks(xs)
    ax.set_xticklabels(RESOLUTIONS)
    ax.set_yticks([])
    ax.set_ylim(6.6, 0.4)
    ax.set_xlim(-2.3, 3.35)
    ax.set_xlabel("evaluation resolution")
    ax.set_ylabel("rank (1 = best)")
    tidy(ax, grid_axis="y")
    ax.grid(axis="y", visible=False)
    ax.spines["left"].set_visible(False)
    save(fig, "seg_fig3_bump")


fig_bump()


## Figure 4 -- parameters against accuracy

Capacity does not order the models: the 7.9 M U-Net is above the 61 M DeepLabV3 at both
ends of the range, and at 256 px it is above every model with more parameters.


In [ ]:
def fig_params():
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
    for ax, r in zip(axes, (256, 2048)):
        col = MIOU[:, RESOLUTIONS.index(r)]
        gap = 0.006 if r == 256 else 0.0012
        label_y = dict(zip(MODELS, spread_labels(list(col), gap)))
        for i, mo in enumerate(MODELS):
            x = PARAMS[mo]
            ax.scatter(x, col[i], s=95, color=FAM_COLOR[FAMILY[mo]], edgecolor=SURFACE,
                       linewidth=1.4, zorder=4)
            ax.annotate(TINY[mo], (x, label_y[mo]), xytext=(9, 0), textcoords="offset points",
                        fontsize=8, va="center", ha="left", color=INK2)
            if abs(label_y[mo] - col[i]) > gap * 0.2:
                ax.annotate("", (x, col[i]), xytext=(x, label_y[mo]),
                            arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        ax.set_xscale("log")
        ax.set_xlim(2.6, 260)
        ax.set_xticks([4, 10, 25, 60])
        ax.set_xticklabels(["4", "10", "25", "60"])
        ax.set_xlabel("parameters (M, log scale)")
        tidy(ax, grid_axis="both")
        ax.set_title(f"{r} px")
        # The two panels carry different y scales on purpose (the 2048 differences would
        # otherwise be invisible); state the spread so the scales cannot mislead.
        ax.text(0.97, 0.97, f"spread {np.ptp(col):.3f} mIoU", transform=ax.transAxes,
                ha="right", va="top", fontsize=8, color=INK2)
    axes[0].set_ylabel("test mIoU")
    axes[0].set_ylim(0.780, 0.935)
    axes[1].set_ylim(0.962, 0.983)
    handles = [Line2D([], [], marker="o", ls="", color=c, label=f, markersize=8)
               for f, c in FAM_COLOR.items()]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save(fig, "seg_fig4_params")


fig_params()


## Figure 5 -- which class pays for low resolution

Building and road are near-ceiling at every level. Cars are not, and the loss is not
only the model's: the downsampled label itself drops from 1.88 % to 1.44 % car pixels
between 2048 and 256, because a car is 2-4 px wide at 256 px and the resampling fuses
neighbours.


In [ ]:
# Value ramp for the terrain panel: red at the bottom of the scale, blue at the top, light
# in the middle so the hillshade still reads. Deliberately not the single-hue blue used for
# the bar and heatmap magnitudes -- this one has to separate the two ends of the range at a
# glance, because the shape of the surface is the message.
VALUE_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "symtrs_value", ["#a3281f", "#d1633f", "#f0d8c4", "#f2f0ec", "#9ec5f4", "#3987e5", "#0d366b"])


def _catmull_rom(k, n):
    """Smooth (Catmull-Rom) resampling matrix from k to n points -- scipy-free."""
    m = np.zeros((n, k))
    for row, x in enumerate(np.linspace(0, k - 1, n)):
        i = min(int(np.floor(x)), k - 2)
        t = x - i
        for d, w in zip((-1, 0, 1, 2),
                        ((-t ** 3 + 2 * t ** 2 - t) / 2, (3 * t ** 3 - 5 * t ** 2 + 2) / 2,
                         (-3 * t ** 3 + 4 * t ** 2 + t) / 2, (t ** 3 - t ** 2) / 2)):
            m[row, min(max(i + d, 0), k - 1)] += w
    return m


def value_terrain(ax, grid, zmin, zmax, zticks, levels, zoom=1.35):
    """A model x resolution grid as a hillshaded surface, with the measured points on it.

    The surface is a cubic interpolation of the measured grid, drawn only so it can be lit;
    every measured value is marked on top so the smoothing cannot pass for data. The camera
    sits over the low corner (256 px, last model) on purpose: that corner is where the rise
    is, and a front-on view hides it behind the plateau.
    """
    from matplotlib.colors import LightSource

    n = 110
    fine = _catmull_rom(len(MODELS), n) @ grid @ _catmull_rom(len(RESOLUTIONS), n).T
    X, Y = np.meshgrid(np.linspace(0, len(RESOLUTIONS) - 1, n),
                       np.linspace(0, len(MODELS) - 1, n))
    light = LightSource(azdeg=312, altdeg=40)
    face = light.shade(fine, cmap=VALUE_CMAP, blend_mode="soft", vert_exag=1.0, dx=0.03, dy=0.05)
    ax.plot_surface(X, Y, fine, facecolors=face, rstride=1, cstride=1, antialiased=False,
                    shade=False, linewidth=0)
    ax.contour(X, Y, fine, levels=levels, zdir="z", offset=zmin,
               colors=[AXIS], linewidths=0.6, alpha=0.85)
    for j in range(len(RESOLUTIONS)):
        for i in range(len(MODELS)):
            ax.plot([j, j], [i, i], [zmin, grid[i, j]], color=AXIS, lw=0.45, alpha=0.5)
            ax.scatter([j], [i], [grid[i, j]], s=11, color=INK, edgecolor=SURFACE,
                       linewidth=0.6, depthshade=False, zorder=20)
    ax.set_box_aspect((1.0, 1.15, 0.66), zoom=zoom)
    ax.set_xticks(range(len(RESOLUTIONS)))
    ax.set_xticklabels([str(r) for r in RESOLUTIONS], fontsize=8.5)
    ax.set_yticks(range(len(MODELS)))
    ax.set_yticklabels([TINY[m] for m in MODELS], fontsize=7.5)
    ax.set_zlim(zmin, zmax)
    ax.set_zticks(zticks)
    ax.tick_params(pad=0)
    # No axis labels: the ticks carry the scales, and at this size an axis label is the one
    # piece of text that gets clipped (this is the panel the title has to name).
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.view_init(elev=26, azim=-128)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor((1, 1, 1, 0))
        axis.line.set_color(AXIS)
    ax.grid(False)


def fig_perclass():
    """(a) IoU for each of the four classes, six models a panel. (b) mean IoU as a surface.

    Part (a) is the class story: building, vegetation and road sit on the ceiling at every
    level, cars do not (0.29 for DeepLabV3 against 0.72 for U-Net at 256 px, closing to
    within 0.02 by 2048). Part (b) reads the same 24 runs as one number each, model against
    resolution: the ordering is stable at the top, and the spread is a 256 px phenomenon
    (0.118 mIoU at 256, 0.012 at 2048).
    """
    # 6.2 in wide because the paper prints it at 5.1 in: drawn at 7.6 it would be
    # shrunk to 68 % and its 10 pt labels would print at 6.8 pt.
    fig = plt.figure(figsize=(6.3, 3.5))
    # The empty third column is a gutter: the 3D panel draws its model labels to the left of
    # its box, and at print width they landed on the car panel's "2048" tick without it.
    gs = fig.add_gridspec(2, 4, left=0.075, right=0.99, top=0.90, bottom=0.155,
                          wspace=0.40, hspace=0.42, width_ratios=[1, 1, 0.12, 1.2])
    xs = np.array(RESOLUTIONS, dtype=float)

    handles = []
    for k, cls in enumerate(CLASSES):
        ax = fig.add_subplot(gs[k // 2, k % 2])
        key = f"iou_{k + 1}"
        for mo in MODELS:
            (ln,) = ax.plot(xs, [DATA[(mo, r)][key] for r in RESOLUTIONS], color=COLOR[mo],
                            marker="o", ms=4.2, markeredgecolor=SURFACE, markeredgewidth=1.0,
                            lw=1.8, label=SHORT[mo])
            if k == 0:
                handles.append(ln)
        vals = [DATA[(mo, r)][key] for mo in MODELS for r in RESOLUTIONS]
        pad = max(0.015, (max(vals) - min(vals)) * 0.16)
        ax.set_ylim(max(0.0, min(vals) - pad), min(1.005, max(vals) + pad))
        ax.set_xscale("log", base=2)
        ax.set_xticks(xs)
        ax.set_xticklabels(RESOLUTIONS)
        ax.set_title(cls)
        tidy(ax)
        if k % 2 == 0:
            ax.set_ylabel("IoU")

    # (b) the same 24 numbers as a surface. The y axis names the models in the same order
    # the four (a) panels plot them, and the legend below keys the six colours once.
    axb = fig.add_subplot(gs[:, 3], projection="3d")
    value_terrain(axb, MIOU, zmin=0.76, zmax=1.0, zticks=[0.80, 0.90, 1.00],
                  levels=[0.80, 0.85, 0.90, 0.95], zoom=1.5)
    axb.set_title("mean IoU", loc="left", pad=2)
    # One legend for the whole figure, in the band under (a): the six colours are the same
    # six models in every panel, and the 3D panel's own y axis already names them.
    fig.legend(handles=handles, loc="lower left", bbox_to_anchor=(0.075, 0.0), ncol=3,
               fontsize=7.2, handlelength=1.5, columnspacing=1.4, labelspacing=0.25,
               borderaxespad=0.0)

    # (a) / (b) sit on the two blocks, level with the panel titles.
    for ax, letter in ((fig.axes[0], "(a)"), (axb, "(b)")):
        bb = ax.get_position()
        fig.text(bb.x0 - 0.052, bb.y1 + 0.022, letter, fontsize=10, fontweight="bold",
                 color=INK, va="center", ha="left")

    save(fig, "seg_fig5_perclass")


fig_perclass()


## Figure 6 -- the whole table as one heatmap

Compact alternative to figures 1-2 for a page-constrained version: every cell is
annotated, so colour only supports the reading.


In [ ]:
def fig_heatmap():
    fig, ax = plt.subplots(figsize=(5.4, 3.4))
    norm = mcolors.Normalize(vmin=0.78, vmax=0.99)
    im = ax.imshow(MIOU, cmap=SEQ, norm=norm, aspect="auto")
    for i in range(len(MODELS)):
        for j in range(4):
            v = MIOU[i, j]
            ax.text(j, i, f"{v:.4f}", ha="center", va="center", fontsize=8.5,
                    color="white" if v > 0.93 else INK,
                    fontweight="bold" if v == MIOU[:, j].max() else "normal")
    ax.set_xticks(range(4))
    ax.set_xticklabels(RESOLUTIONS)
    ax.set_yticks(range(len(MODELS)))
    ax.set_yticklabels([SHORT[m] for m in MODELS])
    ax.set_xlabel("resolution (px)")
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 6, 1), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)
    cb = fig.colorbar(im, ax=ax, pad=0.02, shrink=0.9)
    cb.set_label("test mIoU", color=INK2)
    cb.outline.set_visible(False)
    save(fig, "seg_fig6_heatmap")


fig_heatmap()


## Qualitative panels

One test tile (`loc442_rot013deg_h00`), inferred at all four resolutions with the
evaluation protocol (`--tile-size 512 --tile-overlap 128`, i.e. whole-image sliding
windows), and by all six 2048 checkpoints. 11.5 % of the tile is car, the densest class
in the split after road, so it is exactly the case where resolution should matter.


In [ ]:
GT_NAME = "seg_day_{r}_loc442_rot013deg_h00_x+061425_y+033598_z+007058.png"
RGB_NAME = "rgb_day_{r}_loc442_rot013deg_h00_x+061425_y+033598_z+007058.png"
CROP = (1472, 384, 512)  # x, y, size at 2048 -- the densest car cluster (34 % car)


def ensure_qual_dir(device="cuda:0"):
    """Rebuild `/tmp/segqual/ladder` if it is gone (it lives in /tmp).

    Reads the test split, never writes to it: one tile is symlinked at each resolution and
    run through `symtrs infer` with the evaluation tiling, for all six checkpoints.
    """
    if all((QUAL / "out" / m / str(r) / RGB_NAME.format(r=r).replace(".png", ".npy")).exists()
           for m in MODELS for r in RESOLUTIONS):
        return
    import subprocess
    import sys
    test = DATA_ROOT / "test"
    sub = {1024: "x2", 512: "x4", 256: "x8"}
    for r in RESOLUTIONS:
        src = test / "rgb_day" if r == 2048 else test / "rgb_day_lr" / sub[r]
        gtd = test / "semantic" if r == 2048 else test / "semantic_lr" / sub[r]
        (QUAL / str(r) / "rgb").mkdir(parents=True, exist_ok=True)
        for folder, name in ((src, RGB_NAME), (gtd, GT_NAME)):
            target = QUAL / str(r) / name.format(r=r)
            if not target.exists():
                target.symlink_to(folder / name.format(r=r))
        link = QUAL / str(r) / "rgb" / RGB_NAME.format(r=r)
        if not link.exists():
            link.symlink_to(QUAL / str(r) / RGB_NAME.format(r=r))
    for model in MODELS:
        for r in RESOLUTIONS:
            subprocess.run([sys.executable, "-m", "symtrs.cli", "infer",
                            "--checkpoint", str(RUNS / f"{model}_{r}" / "weights" / "best.pt"),
                            "--images-dir", str(QUAL / str(r) / "rgb"),
                            "--out-dir", str(QUAL / "out" / model / str(r)),
                            "--resolution", str(r), "--tile-size", "512",
                            "--tile-overlap", "128", "--save-npy", "--device", device],
                           cwd=REPO, check=True)


ensure_qual_dir()


def read_rgb(r):
    return np.asarray(plt.imread(QUAL / str(r) / RGB_NAME.format(r=r)))[..., :3]


def read_gt(r):
    raw = np.asarray(Image.open(QUAL / str(r) / GT_NAME.format(r=r))).astype(np.float32)
    return np.rint(raw / 63).astype(np.uint8)


def read_pred(model, r):
    return np.load(QUAL / "out" / model / str(r) / RGB_NAME.format(r=r).replace(".png", ".npy"))


# Class colours are a qualitative map, not the categorical series palette: they label the
# GT, not a model.
CLASS_COLOR = {1: "#d9d9d4", 2: "#7fb069", 3: "#5c5c5c", 4: "#e34948"}
CLASS_RGBA = np.array([[0, 0, 0, 0],
                       mcolors.to_rgba(CLASS_COLOR[1]), mcolors.to_rgba(CLASS_COLOR[2]),
                       mcolors.to_rgba(CLASS_COLOR[3]), mcolors.to_rgba(CLASS_COLOR[4])])


def colourise(label):
    return CLASS_RGBA[label]


def legend_patches():
    # The building swatch is nearly the surface colour; give every patch an edge so the
    # legend is legible without touching the class colours themselves.
    return [Patch(facecolor=CLASS_COLOR[c], edgecolor=MUTED, linewidth=0.6, label=n)
            for c, n in zip(range(1, 5), CLASSES)]


def iou_of(gt, pr, ignore_absent=True):
    out = []
    for c in range(1, 5):
        g, p = gt == c, pr == c
        u = (g | p).sum()
        if u == 0 or (ignore_absent and g.sum() == 0):
            continue
        out.append((g & p).sum() / u)
    return float(np.mean(out))


def strip(ax, img, edge=HAIR):
    ax.imshow(img, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for sp in ax.spines.values():
        sp.set_edgecolor(edge); sp.set_linewidth(0.8)


In [ ]:
def fig_ladder():
    """Rows = resolution, columns = RGB, GT, U-Net, SegFormer b2 -- the whole tile."""
    fig, axes = plt.subplots(4, 4, figsize=(7.0, 7.1))
    for i, r in enumerate(RESOLUTIONS):
        panes = [read_rgb(r), colourise(read_gt(r)), colourise(read_pred("unet_small", r)),
                 colourise(read_pred("segformer_b2", r))]
        for j, (ax, im) in enumerate(zip(axes[i], panes)):
            strip(ax, im)
            if i == 0:
                ax.set_title(["RGB", "GT", "U-Net", "SegFormer b2"][j])
            if j == 0:
                ax.set_ylabel(f"{r} px", color=INK2)
            x0, y0, s = CROP[0] * r / 2048, CROP[1] * r / 2048, CROP[2] * r / 2048
            ax.add_patch(Rectangle((x0, y0), s, s, fill=False, edgecolor=INK, lw=1.0,
                                   linestyle=(0, (3, 2))))
    fig.legend(handles=legend_patches(), loc="lower center", ncol=4, bbox_to_anchor=(0.5, 0.0))
    fig.subplots_adjust(left=0.055, right=0.999, top=0.962, bottom=0.055,
                        hspace=0.02, wspace=0.02)
    save(fig, "seg_fig7_ladder")


In [ ]:
def fig_zoom():
    """Rows = resolution, columns = RGB, GT, U-Net, SegFormer b2 -- the car cluster."""
    fig, axes = plt.subplots(4, 4, figsize=(6.8, 6.9))
    for i, r in enumerate(RESOLUTIONS):
        k, x0, y0, s = 2048 // r, CROP[0] // (2048 // r), CROP[1] // (2048 // r), CROP[2] // (2048 // r)
        gt = read_gt(r)[y0:y0 + s, x0:x0 + s]
        panes = [read_rgb(r)[y0:y0 + s, x0:x0 + s], colourise(gt),
                 colourise(read_pred("unet_small", r)[y0:y0 + s, x0:x0 + s]),
                 colourise(read_pred("segformer_b2", r)[y0:y0 + s, x0:x0 + s])]
        for j, (ax, im) in enumerate(zip(axes[i], panes)):
            strip(ax, im)
            if i == 0:
                ax.set_title(["RGB", "GT", "U-Net", "SegFormer b2"][j])
            if j == 0:
                ax.set_ylabel(f"{r} px", color=INK2)
    fig.legend(handles=legend_patches(), loc="lower center", ncol=4, bbox_to_anchor=(0.5, 0.0))
    fig.subplots_adjust(left=0.075, right=0.999, top=0.958, bottom=0.055,
                        hspace=0.02, wspace=0.02)
    save(fig, "seg_fig8_zoom")


In [ ]:
def fig_errors():
    """The six 2048 checkpoints as error maps on the same crop: black = misclassified."""
    x0, y0, s = CROP
    gt = read_gt(2048)[y0:y0 + s, x0:x0 + s]
    fig, axes = plt.subplots(2, 4, figsize=(7.6, 4.1))
    axes = axes.ravel()
    strip(axes[0], read_rgb(2048)[y0:y0 + s, x0:x0 + s])
    axes[0].set_title("RGB", fontsize=8.5)
    strip(axes[1], colourise(gt))
    axes[1].set_title("GT", fontsize=8.5)
    for ax, mo in zip(axes[2:], MODELS):
        pr = read_pred(mo, 2048)[y0:y0 + s, x0:x0 + s]
        wrong = (pr != gt) & (gt > 0)
        canvas = np.ones((s, s, 3))
        canvas[wrong] = np.array(mcolors.to_rgb(INK))
        strip(ax, canvas)
        ax.set_title(f"{TINY[mo]} -- mIoU {iou_of(gt, pr):.3f}", fontsize=8.5)
    fig.subplots_adjust(left=0.004, right=0.999, top=0.935, bottom=0.004,
                        hspace=0.17, wspace=0.02)
    save(fig, "seg_fig9_errors2048")


fig_ladder()
fig_zoom()
fig_errors()


## Worst cases -- how each model fails, not which one wins

The 2048 checkpoints are within 0.012 mIoU of each other on the test split, so a figure
that ranks them says almost nothing. The useful qualitative question is how each one
fails. The pool is one 512 px crop per sampled test location and each row shows *that
model's* worst crop in the pool -- `argmin` over the pool, recorded in the manifest.


In [ ]:
WORST = Path("/tmp/segqual/failures")
CROP_PX = 512
POOL_STRIDE = 3     # one tile every third test location
CLASS_FLOOR = 0.01  # a class counts towards the score only above 1 % of the crop


def pool_crops():
    """One content-rich 512 px crop per sampled location, chosen from the labels alone.

    Uniform crops mostly land on building or field, where every model scores 1.00 and the
    "worst" case is decided by a 200-pixel sliver -- the first version of this figure showed
    six empty roofs. So the crop is the 512 px window of each tile with the most car, road
    and vegetation in it: an argmax over the ground truth, never over a model's output, so
    it cannot favour any model.
    """
    files = sorted(p.name for p in (DATA_ROOT / "test" / "rgb_day").glob("rgb_day_2048_*.png"))
    per_loc: dict[str, list[str]] = {}
    for f in files:
        per_loc.setdefault(re.search(r"_(loc\d+)_", f).group(1), []).append(f)
    tiles = [per_loc[loc][0] for loc in sorted(per_loc)[::POOL_STRIDE]]

    out = []
    for tile in tiles:
        raw = np.asarray(Image.open(DATA_ROOT / "test" / "semantic" / tile.replace("rgb_day", "seg_day")))
        lab = np.rint(raw.astype(np.float32) / 63).astype(np.uint8)
        blocks = lab.reshape(32, 64, 32, 64)  # 2048 px = 32 blocks of 64 px
        frac = np.stack([(blocks == c).mean(axis=(1, 3)) for c in range(1, 5)])
        # 512 px window = 8 x 8 blocks; 2048 px holds 4 x 4 of them
        win = np.stack([frac[c].reshape(4, 8, 4, 8).mean(axis=(1, 3)) for c in range(4)])
        score = 3 * win[3] + 0.5 * win[2] + 0.5 * win[1]
        i, j = np.unravel_index(score.argmax(), score.shape)
        out.append({"tile": tile, "x": int(j * CROP_PX), "y": int(i * CROP_PX),
                    "frac": {n: float(win[c, i, j]) for c, n in enumerate(CLASSES)}})
    return out


def ensure_worst_pool(device="cuda:2"):
    """Score all six checkpoints on every crop in the pool; return the manifest.

    Cached in /tmp next to the crops: the images are cut from the read-only test split
    (never written into it) and the inference runs once. `device` is cuda:2 because on this
    box the campaign queue holds 0 and 1.
    """
    manifest_path = WORST / "manifest.json"
    if manifest_path.exists():
        return json.loads(manifest_path.read_text())

    import subprocess
    import sys

    crops, gts = WORST / "crops", WORST / "gt"
    crops.mkdir(parents=True, exist_ok=True)
    gts.mkdir(parents=True, exist_ok=True)
    entries = []
    for c in pool_crops():
        stem = re.sub(r"^rgb_day_2048_", "", c["tile"])[:-4]
        rgb = np.asarray(Image.open(DATA_ROOT / "test" / "rgb_day" / c["tile"]))[..., :3]
        raw = np.asarray(Image.open(DATA_ROOT / "test" / "semantic"
                                    / c["tile"].replace("rgb_day", "seg_day")))
        box = (slice(c["y"], c["y"] + CROP_PX), slice(c["x"], c["x"] + CROP_PX))
        Image.fromarray(rgb[box]).save(crops / f"{stem}.png")
        Image.fromarray(raw[box]).save(gts / f"{stem}.png")
        entries.append({"name": stem, "tile": stem, "frac": c["frac"]})

    for model in MODELS:
        subprocess.run([sys.executable, "-m", "symtrs.cli", "infer",
                        "--checkpoint", str(RUNS / f"{model}_2048" / "weights" / "best.pt"),
                        "--images-dir", str(crops), "--out-dir", str(WORST / "out" / model),
                        "--resolution", "2048", "--tile-size", str(CROP_PX),
                        "--tile-overlap", "0", "--save-npy", "--device", device],
                       cwd=REPO, check=True)

    scores, worst = {}, {}
    used = set()
    for model in MODELS:
        scores[model] = {}
        for e in entries:
            raw = np.asarray(Image.open(gts / f"{e['name']}.png")).astype(np.float32)
            gt = np.rint(raw / 63).astype(np.uint8)
            pr = np.load(WORST / "out" / model / f"{e['name']}.npy")
            scores[model][e["name"]] = crop_iou(gt, pr)
        # Greedy over models: the worst crop the rows above have not used yet. Left
        # unconstrained, all six picked the same shadowed car park -- one failure shown six
        # times, which is a worse figure than six failures.
        available = {k: v for k, v in scores[model].items() if k not in used}
        name = min(available, key=lambda k: (np.isnan(available[k]), available[k]))
        used.add(name)
        worst[model] = {"name": name, "miou": scores[model][name]}

    payload = {"crops": entries, "scores": scores, "worst": worst}
    manifest_path.write_text(json.dumps(payload, indent=1))
    return payload


def crop_iou(gt, pr, floor=CLASS_FLOOR):
    """Mean IoU over the classes that cover at least `floor` of the crop.

    A class covering 300 pixels that the model misses scores 0 and drags the mean down;
    that is not a failure worth showing, and without the floor it decides the ranking.
    """
    vals = []
    for c in range(1, 5):
        g, p = gt == c, pr == c
        if g.mean() < floor:
            continue
        union = (g | p).sum()
        vals.append((g & p).sum() / union if union else 0.0)
    return float(np.mean(vals)) if vals else float("nan")


In [ ]:
def fig_worstcases():
    """One row per model: its worst 512 px crop -- RGB, GT, prediction, errors."""
    man = ensure_worst_pool()
    entries = {e["name"]: e for e in man["crops"]}
    fig, axes = plt.subplots(len(MODELS), 4, figsize=(6.4, 9.2))
    for i, model in enumerate(MODELS):
        name = man["worst"][model]["name"]
        rgb = np.asarray(Image.open(WORST / "crops" / f"{name}.png"))[..., :3]
        raw = np.asarray(Image.open(WORST / "gt" / f"{name}.png")).astype(np.float32)
        gt = np.rint(raw / 63).astype(np.uint8)
        pr = np.load(WORST / "out" / model / f"{name}.npy")
        wrong = (pr != gt) & (gt > 0)
        # Errors over a washed-out grey copy of the image: the black marks stay black, and
        # the reader can still see which structure they sit on.
        grey = rgb.mean(axis=2, keepdims=True)
        canvas = np.repeat(np.clip(0.55 * grey + 0.45, 0, 1), 3, axis=2)
        canvas[wrong] = np.array(mcolors.to_rgb(INK))
        for j, pane in enumerate((rgb, colourise(gt), colourise(pr), canvas)):
            strip(axes[i][j], pane)
            if i == 0:
                axes[i][j].set_title(["RGB", "GT", "prediction", "errors"][j])
        share = 100 * wrong.mean()
        e = entries[name]
        # The crop's provenance goes in the row label, not under the last panel: the rows
        # are nearly touching, and an xlabel there lands on the row below.
        axes[i][0].set_ylabel(f"{SHORT[model]}\n"
                              f"IoU {man['worst'][model]['miou']:.2f} · {share:.0f}% wrong\n"
                              f"{e['tile'].split('_')[0]} · {100 * e['frac']['car']:.1f}% car",
                              fontsize=7.2, color=INK2, labelpad=4)
    fig.legend(handles=legend_patches(), loc="lower center", ncol=4,
               bbox_to_anchor=(0.55, 0.002), fontsize=7.5)
    fig.subplots_adjust(left=0.135, right=0.999, top=0.966, bottom=0.042,
                        hspace=0.03, wspace=0.02)
    save(fig, "seg_fig10_worstcases")


fig_worstcases()


## Figure 12 -- every number as a stem (3D)

The car relief lives in Fig. 5 (fourth panel), so this one is about the mean: every one of
the 24 runs hangs from a common floor and the stem length is the score. No smoothing and
no ranking, which is the point -- the previous terrain drew a surface the data does not
support, and read as a slab.


In [ ]:
def fig_stems3d():
    """Every one of the 24 numbers as a stem -- no smoothing, no ranking implied."""
    z0 = 0.74
    fig = plt.figure(figsize=(6.2, 4.5))
    ax = fig.add_axes((0.02, 0.0, 0.94, 0.9), projection="3d")
    ax.set_box_aspect((1.0, 1.35, 0.6), zoom=1.45)
    norm = mcolors.Normalize(0.78, 0.99)
    for i, _ in enumerate(MODELS):
        for j, r in enumerate(RESOLUTIONS):
            z = MIOU[i, j]
            ax.plot([j, j], [i, i], [z0, z], color=AXIS, lw=0.9, alpha=0.9)
            ax.scatter([j], [i], [z], s=34, color=SEQ(norm(z)), edgecolor=INK, linewidth=0.5,
                       depthshade=False, zorder=20)
    ax.set_xticks(range(len(RESOLUTIONS)))
    ax.set_xticklabels([str(r) for r in RESOLUTIONS], fontsize=8)
    ax.set_yticks(range(len(MODELS)))
    ax.set_yticklabels([TINY[m] for m in MODELS], fontsize=8)
    ax.set_zlim(z0, 0.99)
    ax.set_zticks([0.75, 0.80, 0.85, 0.90, 0.95])
    ax.set_xlabel("resolution (px)", labelpad=6)
    ax.set_zlabel("test mIoU", labelpad=2)
    ax.view_init(elev=26, azim=-56)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor((1, 1, 1, 0))
        axis.line.set_color(AXIS)
    ax.grid(False)
    save(fig, "seg_fig12_stems3d")


fig_stems3d()


In [ ]:
# Numbers the text quotes, printed once so a caption never has to guess.
print("\n--- summary for the text ---")
i_min, i_max = int(MIOU[:, 0].argmin()), int(MIOU[:, 0].argmax())
print(f"mIoU at 256:  {MIOU[i_min, 0]:.4f} ({SHORT[MODELS[i_min]]}) .. "
      f"{MIOU[i_max, 0]:.4f} ({SHORT[MODELS[i_max]]}), spread {np.ptp(MIOU[:, 0]):.4f}")
i_min, i_max = int(MIOU[:, 3].argmin()), int(MIOU[:, 3].argmax())
print(f"mIoU at 2048: {MIOU[i_min, 3]:.4f} ({SHORT[MODELS[i_min]]}) .. "
      f"{MIOU[i_max, 3]:.4f} ({SHORT[MODELS[i_max]]}), spread {np.ptp(MIOU[:, 3]):.4f}")
print(f"mean over the six models: {MIOU[:, 0].mean():.4f} (256) -> {MIOU[:, 3].mean():.4f} (2048)")
for i, mo in enumerate(MODELS):
    print(f"  car IoU {SHORT[mo]:<17} {CAR[i, 0]:.4f} -> {CAR[i, 3]:.4f} (+{CAR[i, 3] - CAR[i, 0]:.4f})")
print(f"U-Net 256->1024 +{MIOU[0, 2] - MIOU[0, 0]:.4f}, 1024->2048 +{MIOU[0, 3] - MIOU[0, 2]:.4f}")
print(f"car pixels: {100 * CAR_FRAC[2048]:.3f}% at 2048 -> {100 * CAR_FRAC[256]:.3f}% at 256")
